# Ingestão Gold: métricas de Data Quality para o painel de BI (Looker)
Squad 1 · Dupla 2 · tabela `ecommerce_enderecos`

**KPIs gerados por este notebook:**

| KPI | Fonte | Delta (camada Gold) | SQL Server (Azure) |
|-----|-------|---------------------|--------------------|
| % de falha por regra, por dia | `squad1/dq_monitoring_logs` | `squad1/gold/ecommerce_enderecos/dq_resumo_por_regra` | `squad1.gold_dq_resumo_por_regra_enderecos` |
| % de registros limpos por tabela, por hora | `squad1/silver/ecommerce_enderecos` | `squad1/gold/ecommerce_enderecos/dq_resumo_por_tabela` | `squad1.gold_dq_resumo_por_tabela_enderecos` |

- **% de falha por regra** = `qtd_falhas / qtd_total` de cada uma das 10 regras, somando os micro-lotes do dia.
- **% de registros limpos** = linhas que não falharam em **nenhuma** regra / total de linhas. Vem da Silver, e não do log,
  porque "limpo" é por linha e o log só guarda contagens por regra (a mesma linha seria contada várias vezes).
- Data e hora vêm do caminho do arquivo de origem (`real-time-data/AAAA/MM/DD/HHMMSS/...`), conforme combinado da squad.
- Colunas iguais às da Gold da dupla 3 (`tabela`, `regra`, `_data`, `_hora`, `qtd_*`, `pct_*`), para o Looker unir as duplas;
  `resumo_por_regra` traz também `severidade`.
- A Gold é recalculada por completo a cada execução (`overwrite`), em caminhos e tabelas exclusivos desta dupla.

In [ ]:
import os
import json
from functools import reduce
from dotenv import load_dotenv
from pyspark.errors import AnalysisException
from pyspark.sql import functions as F

# 1. Parâmetros e credenciais (.env)
load_dotenv(".env")

client_id            = os.getenv("ADLS_CLIENT_ID")
tenant_id            = os.getenv("ADLS_TENANT_ID")
client_secret        = os.getenv("ADLS_CLIENT_SECRET")
storage_account_name = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")
sql_host             = os.getenv("SQL_HOST")
sql_database         = os.getenv("SQL_DATABASE")
sql_username         = os.getenv("SQL_USERNAME")
sql_password         = os.getenv("SQL_PASSWORD")

variaveis = {
    "ADLS_CLIENT_ID": client_id,
    "ADLS_TENANT_ID": tenant_id,
    "ADLS_CLIENT_SECRET": client_secret,
    "SQL_HOST": sql_host,
    "SQL_DATABASE": sql_database,
    "SQL_USERNAME": sql_username,
    "SQL_PASSWORD": sql_password,
}
faltantes = [nome for nome, valor in variaveis.items() if not valor]
assert not faltantes, f"Faltou variável no .env: {', '.join(faltantes)}"

adls_opts = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}
sql_opts = {
    "host": sql_host,
    "port": "1433",
    "database": sql_database,
    "user": sql_username,
    "password": sql_password,
    "encrypt": "true",
    "trustServerCertificate": "false",
}

NOME_TABELA = "ecommerce_enderecos"
squad_base  = f"abfss://squad1@{storage_account_name}.dfs.core.windows.net"
logs_path   = f"{squad_base}/dq_monitoring_logs"
silver_path = f"{squad_base}/silver/{NOME_TABELA}"
gold_base   = f"{squad_base}/gold/{NOME_TABELA}"

# Caminho Delta da Gold -> tabela no SQL Server
DESTINOS_GOLD = {
    "dq_resumo_por_regra":  (f"{gold_base}/dq_resumo_por_regra",  "squad1.gold_dq_resumo_por_regra_enderecos"),
    "dq_resumo_por_tabela": (f"{gold_base}/dq_resumo_por_tabela", "squad1.gold_dq_resumo_por_tabela_enderecos"),
}

# Data e hora de geração do arquivo: real-time-data/AAAA/MM/DD/HHMMSS/...
RE_DATA_HORA = r"real-time-data/(\d{4})/(\d{2})/(\d{2})/(\d{2})\d{4}/"

def extrair_data(coluna):
    return F.concat_ws("-", *[F.regexp_extract(coluna, RE_DATA_HORA, i) for i in (1, 2, 3)])

def extrair_hora(coluna):
    return F.regexp_extract(coluna, RE_DATA_HORA, 4)

def percentual(parte, total):
    return F.when(F.col(total) > 0, F.round(F.col(parte) / F.col(total) * 100, 2))

def ler_delta(path):
    try:
        df = spark.read.format("delta").options(**adls_opts).load(path)
        df.schema
        return df
    except AnalysisException as e:
        if any(m in str(e) for m in ("PATH_NOT_FOUND", "DELTA_PATH_DOES_NOT_EXIST", "DELTA_MISSING_DELTA_TABLE", "is not a Delta table")):
            return None
        raise

### KPI 1: % de falha por regra, por dia
Agrega os registros da `dq_monitoring_logs` desta tabela (uma linha por regra por micro-lote) por regra e dia.

In [ ]:
# 2. KPI 1 — dq_resumo_por_regra
df_logs = ler_delta(logs_path)
assert df_logs is not None, f"dq_monitoring_logs não encontrada em {logs_path}"

df_resumo_regra = (
    df_logs
    .filter(F.col("tabela") == NOME_TABELA)
    .withColumn("_data", extrair_data("arquivo_origem"))
    .groupBy("tabela", "regra", "severidade", "_data")
    .agg(
        F.sum("qtd_registros_falhos").alias("qtd_falhas"),
        F.sum("qtd_registros_total").alias("qtd_total"),
    )
    .withColumn("pct_falha", percentual("qtd_falhas", "qtd_total"))
    .select("tabela", "regra", "_data", "qtd_falhas", "qtd_total", "pct_falha", "severidade")
)
display(df_resumo_regra.orderBy("_data", "regra"))

### KPI 2: % de registros limpos por tabela, por hora
Uma linha da Silver é **limpa** quando todas as colunas `falha_r*` são `False`.

In [ ]:
# 3. KPI 2 — dq_resumo_por_tabela
df_silver = ler_delta(silver_path)
assert df_silver is not None, f"Silver não encontrada em {silver_path}"

colunas_falha = [c for c in df_silver.columns if c.startswith("falha_r")]
assert colunas_falha, "Silver sem colunas falha_r*"
falhou_alguma = reduce(lambda a, b: a | b, [F.coalesce(F.col(c), F.lit(False)) for c in colunas_falha])

df_resumo_tabela = (
    df_silver
    .withColumn("_data", extrair_data("bronze_source_file"))
    .withColumn("_hora", extrair_hora("bronze_source_file"))
    .groupBy("_data", "_hora")
    .agg(
        F.sum(F.when(~falhou_alguma, 1).otherwise(0)).alias("qtd_limpos"),
        F.count("*").alias("qtd_total"),
    )
    .withColumn("tabela", F.lit(NOME_TABELA))
    .withColumn("pct_limpo", percentual("qtd_limpos", "qtd_total"))
    .select("_data", "_hora", "qtd_limpos", "qtd_total", "tabela", "pct_limpo")
)
display(df_resumo_tabela.orderBy("_data", "_hora"))

### Gravação
Delta na camada Gold e espelho no Azure SQL Server, com conferência de contagem após a carga.
Uma falha no SQL Server não desfaz a Delta; o orquestrador tenta de novo na próxima rodada.

In [ ]:
# 4. Gravação Delta (Gold) e SQL Server
resultados = {"dq_resumo_por_regra": df_resumo_regra, "dq_resumo_por_tabela": df_resumo_tabela}
retorno = {"delta": {}, "sql": {}, "falhas": []}

for nome, df in resultados.items():
    delta_path, sql_tabela = DESTINOS_GOLD[nome]
    try:
        (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
           .options(**adls_opts).save(delta_path))
        df_gold = ler_delta(delta_path)
        qtd = df_gold.count()
        retorno["delta"][nome] = qtd
        print(f"✔ Delta {nome}: {qtd} linha(s)")

        df_gold.write.format("sqlserver").options(**sql_opts).option("dbtable", sql_tabela).option("truncate", "true").mode("overwrite").save()
        qtd_sql = spark.read.format("sqlserver").options(**sql_opts).option("dbtable", sql_tabela).load().count()
        retorno["sql"][nome] = qtd_sql
        if qtd_sql != qtd:
            raise ValueError(f"contagem divergente no SQL Server ({qtd_sql}) x Delta ({qtd})")
        print(f"✔ SQL Server {sql_tabela}: {qtd_sql} linha(s)")
    except Exception as e:
        retorno["falhas"].append(nome)
        print(f"✖ Falha em {nome}: {e}")

In [ ]:
# 5. Retorno para o orquestrador
dbutils.notebook.exit(json.dumps(retorno))